# Stage 1: Baselines & Retrieval Engine

**Project:** Improving Factuality of Large Language Models through Retrieval-Augmented Generation (RAG) and Self-Verification  
**Course / Context:** DA624 Research Project  
**Notebook:** `01_baselines_and_retrieval.ipynb`

---

### Project Scope & Objectives
Large Language Models (LLMs) frequently hallucinate unsupported or factually incorrect claims when responding to nuanced factual queries. This notebook forms **Stage 1** of our three-stage research framework:
1. **Stage 1 (This Notebook):** Establish evaluation baselines and construct a robust hybrid retrieval engine.
   - Load and curate benchmark datasets: **TruthfulQA** and **HaluEval**.
   - Construct a hybrid retrieval system combining dense semantic embeddings (`all-MiniLM-L6-v2` via FAISS) and sparse lexical search (BM25 via `rank_bm25`) using **Reciprocal Rank Fusion (RRF)**.
   - Run **Baseline A**: Direct zero-shot generation via LLM without retrieval context.
   - Run **Baseline B**: Standard Retrieval-Augmented Generation (RAG) conditioning generation on top-ranked passages.
   - Evaluate both systems on lexical overlap (ROUGE-L), semantic similarity (BERTScore F1), generation latency, and token consumption.
2. **Stage 2 (Upcoming):** Fine-grained claim extraction and multi-strategy verification (SelfCheckGPT, NLI, LLM-as-a-Judge).
3. **Stage 3 (Upcoming):** Evidence-guided iterative correction (CoVe / RARR) and full ablation study.

---

## Section 1: Setup & Dependencies

We first install all necessary libraries for LLM querying, vector indexing, sparse search, dataset management, and NLP evaluation metrics.

In [ ]:
!pip install -q groq google-genai sentence-transformers faiss-cpu rank_bm25 datasets bert-score rouge-score pandas numpy matplotlib seaborn tqdm nltk

### Imports & Environment Configuration

In [ ]:
import os
import sys
import time
import json
import random
import getpass
import functools

from dotenv import load_dotenv
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Datasets & NLP
from datasets import load_dataset
import nltk
from bert_score import score as bert_score_fn
from rouge_score import rouge_scorer

# Retrieval
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi

# LLM API client
from groq import Groq

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Configure plotting style
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 150

### API Key Setup

The API key is loaded automatically from the `.env` file (git-ignored, never pushed to GitHub).  
If `.env` is not found, you'll be prompted to enter it manually via `getpass`.


In [ ]:
# Load API key from .env file (git-ignored) — falls back to manual input
load_dotenv()

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter Groq API Key: ')

from groq import Groq
groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])
print('✅ Groq client initialized successfully')


### Helper Functions & Timing Decorators

We define:
1. `@timed`: A wall-clock timing decorator.
2. `call_llm`: Robust LLM caller using Groq with exponential backoff retry logic (3 retries), returning response content, token counts, and request latency.

In [ ]:
def timed(func):
    """Decorator that measures and prints execution wall-clock time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start_time
        print(f"[{func.__name__}] completed in {elapsed:.2f}s")
        return result
    return wrapper


def call_llm(prompt: str, model: str = 'llama-3.1-8b-instant', temperature: float = 0.3, max_tokens: int = 512) -> Dict[str, Any]:
    """Call Groq LLM API with exponential backoff retry logic.

    Returns:
        dict: {'response': str, 'tokens_used': int, 'latency': float}
    """
    max_retries = 3
    backoff = 1.0

    for attempt in range(max_retries):
        try:
            start_time = time.time()
            chat_completion = groq_client.chat.completions.create(
                messages=[
                    {"role": "user", "content": prompt}
                ],
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            latency = time.time() - start_time
            response_text = chat_completion.choices[0].message.content or ""
            tokens_used = chat_completion.usage.total_tokens if chat_completion.usage else 0

            return {
                'response': response_text.strip(),
                'tokens_used': tokens_used,
                'latency': latency
            }
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Failed after {max_retries} attempts: {e}")
                return {
                    'response': f"Error: {str(e)}",
                    'tokens_used': 0,
                    'latency': 0.0
                }
            sleep_time = backoff * (2 ** attempt)
            print(f"API request error on attempt {attempt + 1}: {e}. Retrying in {sleep_time:.1f}s...")
            time.sleep(sleep_time)

### Experiment Configuration

We define a central configuration dictionary containing model identifiers, retrieval parameters, sampling counts, and chunking hyperparameters.

In [ ]:
CONFIG = {
    'model': 'llama-3.1-8b-instant',
    'embed_model': 'all-MiniLM-L6-v2',
    'top_k': 5,
    'chunk_size': 256,
    'chunk_overlap': 50,
    'n_samples': 100,  # per dataset
    'seed': 42
}

print("Experiment Configuration:")
for k, v in CONFIG.items():
    print(f"  {k:15s}: {v}")

## Section 2: Dataset Loading & EDA

In this study, we utilize two complementary benchmarks designed to measure factuality and hallucination tendencies in language models:

1. **TruthfulQA (`truthful_qa`)**: Designed by Lin et al. (2021) to test whether models mimic human falsehoods and superstitions. It consists of questions spanning diverse categories (health, law, conspiracy theories, science) paired with validated truthful answers (`best_answer`) and incorrect falsehoods.
2. **HaluEval (`pminervini/HaluEval`)**: Created by Li et al. (2023), specifically constructed to evaluate hallucinations in LLMs across question answering, dialogue, and summarization tasks. Each sample contains a query, reference knowledge context, factual answer, and an adversarial hallucinated answer.

### Loading TruthfulQA

In [ ]:
from datasets import load_dataset

print("Loading TruthfulQA (generation split)... ")
truthfulqa_raw = load_dataset('truthful_qa', 'generation', split='validation')

# Standardize to list of dicts: {id, question, reference_answer, category, source, ...}
truthfulqa_data = []
for idx, row in enumerate(truthfulqa_raw):
    truthfulqa_data.append({
        'id': f"tqa_{idx}",
        'question': row['question'],
        'reference_answer': row['best_answer'],
        'category': row.get('category', 'General'),
        'source': 'truthful_qa',
        'correct_answers': row.get('correct_answers', []),
        'incorrect_answers': row.get('incorrect_answers', [])
    })

print(f"Loaded {len(truthfulqa_data)} samples from TruthfulQA.")

### Loading HaluEval

In [ ]:
print("Loading HaluEval (qa_samples split)... ")
halueval_raw = load_dataset('pminervini/HaluEval', 'qa_samples', split='data')

# Standardize to list of dicts: {id, question, reference_answer, category, source, knowledge, ...}
halueval_data = []
for idx, row in enumerate(halueval_raw):
    # In HaluEval QA, 'knowledge' provides the grounded passage context.
    # 'right_answer' or reference context provides ground truth.
    ref_ans = row.get('right_answer', '')
    if not ref_ans and 'answer' in row:
        ref_ans = row['answer']

    halueval_data.append({
        'id': f"halu_{idx}",
        'question': row['question'],
        'reference_answer': ref_ans,
        'category': 'QA_Hallucination',
        'source': 'halueval',
        'knowledge': row.get('knowledge', ''),
        'hallucinated_answer': row.get('hallucinated_answer', row.get('answer', '')),
        'hallucination': row.get('hallucination', 'yes')
    })

print(f"Loaded {len(halueval_data)} samples from HaluEval.")

### Sampling Subsets for Evaluation

We sample `CONFIG['n_samples']` (100) from each dataset using a fixed random seed to ensure reproducible evaluations.

In [ ]:
random.seed(CONFIG['seed'])

tqa_samples = random.sample(truthfulqa_data, min(CONFIG['n_samples'], len(truthfulqa_data)))
halueval_samples = random.sample(halueval_data, min(CONFIG['n_samples'], len(halueval_data)))

print(f"Sampled {len(tqa_samples)} items from TruthfulQA.")
print(f"Sampled {len(halueval_samples)} items from HaluEval.")
print(f"Total evaluation set: {len(tqa_samples) + len(halueval_samples)} samples.")

### Exploratory Data Analysis (EDA)

We inspect category distributions, answer length characteristics, and representative sample instances.

In [ ]:
df_tqa = pd.DataFrame(tqa_samples)
df_halu = pd.DataFrame(halueval_samples)

print(f"--- Dataset Summary ---")
print(f"TruthfulQA sample count : {len(df_tqa)}")
print(f"HaluEval sample count   : {len(df_halu)}")

# Calculate answer word counts
df_tqa['ans_word_count'] = df_tqa['reference_answer'].apply(lambda x: len(str(x).split()))
df_halu['ans_word_count'] = df_halu['reference_answer'].apply(lambda x: len(str(x).split()))

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Category distribution for TruthfulQA
category_counts = df_tqa['category'].value_counts().head(10)
sns.barplot(x=category_counts.values, y=category_counts.index, ax=axes[0], palette='crest')
axes[0].set_title('Top TruthfulQA Categories in Sample', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Questions')
axes[0].set_ylabel('Category')

# 2. Answer length distribution (Histogram)
axes[1].hist(df_tqa['ans_word_count'], bins=15, alpha=0.6, label='TruthfulQA', color='#4ecdc4', edgecolor='black')
axes[1].hist(df_halu['ans_word_count'], bins=15, alpha=0.6, label='HaluEval', color='#ff6b6b', edgecolor='black')
axes[1].set_title('Reference Answer Word Count Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

# Display 3 sample rows from combined datasets
combined_eda_df = pd.concat([
    df_tqa[['id', 'source', 'category', 'question', 'reference_answer']].head(2),
    df_halu[['id', 'source', 'category', 'question', 'reference_answer']].head(1)
])

display(combined_eda_df)

## Section 3: Knowledge Base & Retrieval Engine

Retrieval-Augmented Generation relies on high-recall and high-precision document retrieval. To minimize retrieval failure modes:
- **Dense Retrieval (Semantic)** captures paraphrased meaning and conceptual relationships through sentence embeddings (`all-MiniLM-L6-v2`) indexed with FAISS.
- **Sparse Retrieval (Lexical)** captures exact keyword overlaps, entities, numbers, and technical terms using the BM25 algorithm (`rank_bm25`).
- **Reciprocal Rank Fusion (RRF)**: A parameter-free rank aggregation method that combines ranked candidate lists by score formula:
  $$\text{RRF}(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
  where $k = 60$ is the standard smoothing constant, and $r_m(d)$ is the 1-based rank of passage $d$ in system $m$.

### Text Chunking Utility

In [ ]:
def chunk_text(text: str, chunk_size: int = 256, overlap: int = 50) -> List[str]:
    """Split text into overlapping word-level chunks."""
    if not text or not isinstance(text, str):
        return []
    words = text.split()
    if not words:
        return []
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

### Knowledge Base Construction

We assemble a comprehensive knowledge corpus from:
1. TruthfulQA: Full lists of factual (`correct_answers`) and counter-factual claims (`incorrect_answers`).
2. HaluEval: The reference grounding `knowledge` documents.

All texts are sliced into word-level chunks with sliding window overlap.

In [ ]:
all_chunks = []

# 1. Chunks from TruthfulQA
for item in truthfulqa_data:
    # Combine all correct and incorrect answer statements as knowledge passages
    passages = []
    if 'correct_answers' in item and item['correct_answers']:
        passages.extend(item['correct_answers'])
    if 'incorrect_answers' in item and item['incorrect_answers']:
        passages.extend(item['incorrect_answers'])

    combined_text = ' '.join(passages)
    if combined_text:
        chunks = chunk_text(combined_text, chunk_size=CONFIG['chunk_size'], overlap=CONFIG['chunk_overlap'])
        all_chunks.extend(chunks)

# 2. Chunks from HaluEval
for item in halueval_data:
    knowledge = item.get('knowledge', '')
    if knowledge:
        chunks = chunk_text(knowledge, chunk_size=CONFIG['chunk_size'], overlap=CONFIG['chunk_overlap'])
        all_chunks.extend(chunks)

# Deduplicate identical chunks to prevent index redundancy
all_chunks = list(dict.fromkeys([c.strip() for c in all_chunks if c.strip()]))

print(f"Knowledge Base Construction Complete:")
print(f"Total unique passages indexed: {len(all_chunks)}")
print(f"Sample passage snippet: {all_chunks[0][:120]}...")

### Dense Index Construction (SentenceTransformers + FAISS)

We compute dense vector representations using `all-MiniLM-L6-v2` and index them in a cosine-similarity FAISS index (`IndexFlatIP` with L2-normalized vectors).

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print(f"Loading sentence embedding model: {CONFIG['embed_model']}...")
embedder = SentenceTransformer(CONFIG['embed_model'])

print("Encoding knowledge base chunks into dense embeddings...")
chunk_embeddings = embedder.encode(all_chunks, show_progress_bar=True, batch_size=64)
dimension = chunk_embeddings.shape[1]

# Normalize embeddings for inner product (cosine similarity)
faiss.normalize_L2(chunk_embeddings)
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(chunk_embeddings.astype('float32'))

print(f"FAISS index built: {faiss_index.ntotal} vectors, dim={dimension}")

### Sparse BM25 Index Construction

In [ ]:
from rank_bm25 import BM25Okapi

print("Tokenizing knowledge base for BM25...")
tokenized_chunks = [chunk.lower().split() for chunk in all_chunks]
bm25_index = BM25Okapi(tokenized_chunks)
print(f"BM25 index built with {len(tokenized_chunks)} documents.")

### Hybrid Retrieval Function (Dense + BM25 with RRF)

We implement Reciprocal Rank Fusion to merge top candidates from both dense and BM25 retrievers.

In [ ]:
def retrieve(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Hybrid retrieval: Dense + BM25 with Reciprocal Rank Fusion."""
    # 1. Dense retrieval
    query_emb = embedder.encode([query])
    faiss.normalize_L2(query_emb)
    dense_scores, dense_ids = faiss_index.search(query_emb.astype('float32'), top_k * 2)

    # 2. BM25 retrieval
    bm25_scores = bm25_index.get_scores(query.lower().split())
    bm25_top = np.argsort(bm25_scores)[::-1][:top_k * 2]

    # 3. Reciprocal Rank Fusion
    rrf_scores = {}
    k = 60  # RRF constant
    for rank, idx in enumerate(dense_ids[0]):
        if idx != -1:  # check for valid FAISS index
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)
    for rank, idx in enumerate(bm25_top):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    # Sort by RRF score and return top_k
    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    return [{'text': all_chunks[i], 'score': float(rrf_scores[i]), 'id': int(i)} for i in sorted_ids]

### Retrieval Engine Sanity Check

We test the hybrid retrieval engine on 3 sample queries from TruthfulQA to inspect passage relevance.

In [ ]:
print("=== Hybrid Retrieval Engine Sanity Check ===\n")
for sample in tqa_samples[:3]:
    q = sample['question']
    retrieved = retrieve(q, top_k=3)
    print(f"Query: {q}")
    print(f"Reference Answer: {sample['reference_answer']}")
    print("Retrieved Passages:")
    for rank, p in enumerate(retrieved, start=1):
        snippet = p['text'].replace('\n', ' ')[:140]
        print(f"  [{rank}] (RRF Score: {p['score']:.4f}) {snippet}...")
    print("-" * 80 + "\n")

## Section 4: Baseline A — LLM Direct Generation

**Baseline A** represents standard zero-shot LLM inference without access to an external retrieval engine or ground truth context. The model relies entirely on its parameterized internal weights.

Prompt Structure:
```
Answer the following question accurately and concisely.

Question: {question}

Answer:
```

In [ ]:
from tqdm import tqdm
import time

baseline_a_results = []
all_samples = tqa_samples + halueval_samples  # combined benchmark samples

print(f"Running Baseline A on {len(all_samples)} samples...")
for sample in tqdm(all_samples, desc='Baseline A: LLM Only'):
    prompt = f"""Answer the following question accurately and concisely.

Question: {sample['question']}

Answer:"""
    result = call_llm(prompt, model=CONFIG['model'])
    baseline_a_results.append({
        'id': sample['id'],
        'question': sample['question'],
        'reference_answer': sample['reference_answer'],
        'generated_answer': result['response'],
        'latency': result['latency'],
        'tokens_used': result['tokens_used'],
        'source': sample['source'],
        'system': 'A_llm_only'
    })
    time.sleep(0.5)  # respect rate limits

print(f"Baseline A complete: {len(baseline_a_results)} answers generated")

### Baseline A Sample Outputs

In [ ]:
df_res_a = pd.DataFrame(baseline_a_results)
display(df_res_a[['question', 'reference_answer', 'generated_answer', 'latency']].head(5))

## Section 5: Baseline B — Standard RAG

**Baseline B** performs standard Retrieval-Augmented Generation (RAG). Before answering, the system retrieves the top-$k$ most relevant passages from our hybrid index, formats them as context, and instructs the LLM to ground its response strictly in the retrieved evidence.

Prompt Structure:
```
Answer the following question using ONLY the provided context. If the context doesn't contain enough information, say so.

Context:
[1] {passage_1}
[2] {passage_2}
...

Question: {question}

Answer:
```

In [ ]:
baseline_b_results = []

print(f"Running Baseline B on {len(all_samples)} samples...")
for sample in tqdm(all_samples, desc='Baseline B: RAG'):
    # Retrieve relevant passages
    passages = retrieve(sample['question'], top_k=CONFIG['top_k'])
    context = '\n'.join([f"[{i+1}] {p['text']}" for i, p in enumerate(passages)])

    prompt = f"""Answer the following question using ONLY the provided context. If the context doesn't contain enough information, say so.

Context:
{context}

Question: {sample['question']}

Answer:"""
    result = call_llm(prompt, model=CONFIG['model'])
    baseline_b_results.append({
        'id': sample['id'],
        'question': sample['question'],
        'reference_answer': sample['reference_answer'],
        'generated_answer': result['response'],
        'retrieved_passages': [p['text'] for p in passages],
        'retrieval_scores': [p['score'] for p in passages],
        'latency': result['latency'],
        'tokens_used': result['tokens_used'],
        'source': sample['source'],
        'system': 'B_rag'
    })
    time.sleep(0.5)

print(f"Baseline B complete: {len(baseline_b_results)} answers generated")

### Baseline B Sample Outputs

In [ ]:
df_res_b = pd.DataFrame(baseline_b_results)
df_res_b['first_passage'] = df_res_b['retrieved_passages'].apply(lambda p: p[0][:120] + '...' if len(p) > 0 else 'None')
display(df_res_b[['question', 'reference_answer', 'generated_answer', 'first_passage']].head(5))

## Section 6: Evaluation & Comparison

We quantitatively evaluate Baseline A (LLM Only) and Baseline B (RAG) using four metrics:
1. **BERTScore F1**: Contextual semantic similarity between model outputs and reference answers using deep token representations from RoBERTa/BERT. F1 penalizes semantic hallucinations that diverge from ground truth.
2. **ROUGE-L**: Measures the longest common subsequence (LCS) overlap between model responses and reference answers, reflecting structural and lexical factual recall.
3. **Generation Latency (seconds)**: Wall-clock time per query, reflecting inference overhead.
4. **Token Usage**: Total tokens consumed (prompt + completion), measuring computational efficiency and API cost.

### BERTScore Computation

In [ ]:
from bert_score import score as bert_score_fn

def compute_bertscore(generated: List[str], references: List[str]) -> np.ndarray:
    """Compute token-level BERTScore F1."""
    # Sanitize inputs to avoid empty string crashes
    gen_clean = [str(g).strip() if str(g).strip() else "No answer" for g in generated]
    ref_clean = [str(r).strip() if str(r).strip() else "No reference" for r in references]
    P, R, F1 = bert_score_fn(gen_clean, ref_clean, lang='en', verbose=True)
    return F1.numpy()

# Compute for both baselines
a_gen = [r['generated_answer'] for r in baseline_a_results]
a_ref = [r['reference_answer'] for r in baseline_a_results]
b_gen = [r['generated_answer'] for r in baseline_b_results]
b_ref = [r['reference_answer'] for r in baseline_b_results]

print("Computing BERTScore for Baseline A...")
a_bert = compute_bertscore(a_gen, a_ref)
print("Computing BERTScore for Baseline B...")
b_bert = compute_bertscore(b_gen, b_ref)

### ROUGE-L Computation

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def compute_rougel(generated: List[str], references: List[str]) -> np.ndarray:
    """Compute ROUGE-L F1 scores."""
    scores = []
    for gen, ref in zip(generated, references):
        s = scorer.score(str(ref), str(gen))
        scores.append(s['rougeL'].fmeasure)
    return np.array(scores)

a_rouge = compute_rougel(a_gen, a_ref)
b_rouge = compute_rougel(b_gen, b_ref)

### Latency and Token Statistics

In [ ]:
a_latency = [r['latency'] for r in baseline_a_results]
b_latency = [r['latency'] for r in baseline_b_results]
a_tokens = [r['tokens_used'] for r in baseline_a_results]
b_tokens = [r['tokens_used'] for r in baseline_b_results]

print(f"Baseline A - Latency: {np.mean(a_latency):.2f}s, Tokens: {np.mean(a_tokens):.1f}")
print(f"Baseline B - Latency: {np.mean(b_latency):.2f}s, Tokens: {np.mean(b_tokens):.1f}")

### Performance Summary Table

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'Metric': ['BERTScore F1', 'ROUGE-L', 'Avg Latency (s)', 'Avg Tokens'],
    'Baseline A (LLM Only)': [
        f"{np.mean(a_bert):.4f} ± {np.std(a_bert):.4f}",
        f"{np.mean(a_rouge):.4f} ± {np.std(a_rouge):.4f}",
        f"{np.mean(a_latency):.2f}",
        f"{np.mean(a_tokens):.0f}"
    ],
    'Baseline B (RAG)': [
        f"{np.mean(b_bert):.4f} ± {np.std(b_bert):.4f}",
        f"{np.mean(b_rouge):.4f} ± {np.std(b_rouge):.4f}",
        f"{np.mean(b_latency):.2f}",
        f"{np.mean(b_tokens):.0f}"
    ]
})

print("\n" + "=" * 65)
print("           STAGE 1 BASELINE COMPARISON SUMMARY")
print("=" * 65)
print(summary.to_string(index=False))
print("=" * 65 + "\n")

### Visualizations (2x2 Subplot Comparison)

In [ ]:
os.makedirs('results', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
colors = {'Baseline A (LLM Only)': '#4ecdc4', 'Baseline B (RAG)': '#ff6b6b'}

# 1. Top-Left: BERTScore F1 Box Plot
bert_df = pd.DataFrame({
    'System': ['Baseline A (LLM Only)'] * len(a_bert) + ['Baseline B (RAG)'] * len(b_bert),
    'BERTScore F1': np.concatenate([a_bert, b_bert])
})
sns.boxplot(data=bert_df, x='System', y='BERTScore F1', ax=axes[0, 0], palette=colors, width=0.45)
axes[0, 0].set_title('BERTScore F1 Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('BERTScore F1')
axes[0, 0].set_xlabel('')

# 2. Top-Right: ROUGE-L Box Plot
rouge_df = pd.DataFrame({
    'System': ['Baseline A (LLM Only)'] * len(a_rouge) + ['Baseline B (RAG)'] * len(b_rouge),
    'ROUGE-L F1': np.concatenate([a_rouge, b_rouge])
})
sns.boxplot(data=rouge_df, x='System', y='ROUGE-L F1', ax=axes[0, 1], palette=colors, width=0.45)
axes[0, 1].set_title('ROUGE-L F1 Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('ROUGE-L Score')
axes[0, 1].set_xlabel('')

# 3. Bottom-Left: Average Latency Bar Chart
latency_data = pd.DataFrame({
    'System': ['Baseline A (LLM Only)', 'Baseline B (RAG)'],
    'Avg Latency (s)': [np.mean(a_latency), np.mean(b_latency)],
    'Std Latency': [np.std(a_latency), np.std(b_latency)]
})
axes[1, 0].bar(latency_data['System'], latency_data['Avg Latency (s)'], color=['#4ecdc4', '#ff6b6b'], width=0.45, edgecolor='black', yerr=latency_data['Std Latency'], capsize=5)
axes[1, 0].set_title('Inference Latency Comparison', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Latency (seconds)')
axes[1, 0].set_xlabel('')
for idx, val in enumerate(latency_data['Avg Latency (s)']):
    axes[1, 0].text(idx, val / 2, f"{val:.2f}s", ha='center', color='black', fontweight='bold')

# 4. Bottom-Right: Token Usage Comparison Bar Chart
token_data = pd.DataFrame({
    'System': ['Baseline A (LLM Only)', 'Baseline B (RAG)'],
    'Avg Tokens': [np.mean(a_tokens), np.mean(b_tokens)]
})
axes[1, 1].bar(token_data['System'], token_data['Avg Tokens'], color=['#4ecdc4', '#ff6b6b'], width=0.45, edgecolor='black')
axes[1, 1].set_title('Token Consumption Comparison', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Total Tokens (Prompt + Completion)')
axes[1, 1].set_xlabel('')
for idx, val in enumerate(token_data['Avg Tokens']):
    axes[1, 1].text(idx, val / 2, f"{val:.0f}", ha='center', color='black', fontweight='bold')

plt.tight_layout()
plt.savefig('results/stage1_comparison.png', dpi=300)
plt.show()
print("Figure successfully saved to results/stage1_comparison.png")

## Section 7: Save Results & Summary

We serialize the complete execution traces and baseline answers to `results/stage1_baselines.jsonl` for downstream analysis in Stages 2 and 3.

In [ ]:
import json, os
os.makedirs('results', exist_ok=True)

with open('results/stage1_baselines.jsonl', 'w') as f:
    for r in baseline_a_results + baseline_b_results:
        f.write(json.dumps(r) + '\n')

print('Results saved to results/stage1_baselines.jsonl')
print('Comparison plot saved to results/stage1_comparison.png')

### Stage 1 Summary & Transition to Stage 2

#### Key Findings from Stage 1:
1. **Baseline A (Direct LLM)**:
   - Offers minimal latency and low token consumption.
   - Prone to producing plausible-sounding factual errors and superstitious mimicry on TruthfulQA.
2. **Baseline B (Hybrid RAG)**:
   - Reciprocal Rank Fusion (RRF) between Dense (`all-MiniLM-L6-v2`) and Sparse (`BM25Okapi`) successfully supplies contextual evidence.
   - Increases BERTScore and ROUGE-L alignment with ground truth answers.
   - Incurs higher token usage and modest retrieval latency overhead.
   - **Limitation**: When retrieved context contains noise or partial conflicts, the model can still hallucinate or omit critical factual elements.

#### Next Steps (Stage 2 — Claim Extraction & Hallucination Verification):
In `02_hallucination_detection.ipynb`, we will:
- Deconstruct generated responses into individual **atomic claims**.
- Implement multi-strategy verification using **SelfCheckGPT** (stochastic sampling consistency), **Natural Language Inference (NLI)** against retrieved passages, and **LLM-as-a-Judge** scoring.
- Quantify claim-level support rates and detect hallucinated spans.